# Apache Kafka — First Contact

Kafka is a distributed event streaming platform built to handle continuous flows of data at high throughput with strong durability. Instead of treating data as isolated rows written directly into downstream systems, Kafka treats events as an ordered log that can be written once and read many times by independent consumers.

Kafka exists because tightly coupling producers to databases, services, and analytics pipelines creates bottlenecks and fragile dependencies. In a modern data engineering stack, Kafka sits between event producers and downstream consumers so that ingestion, alerting, analytics, and machine learning pipelines can scale independently.

In Citi's telemetry system, 6,000+ endpoints emit latency and error events continuously. A database cannot absorb that write rate without becoming a bottleneck. Kafka decouples producers (monitoring agents) from consumers (alerting, analytics, ML pipelines).

```text
                 +------------------+
                 | Monitoring Agent |
                 +------------------+
                          |
                          v
                    +-------------+
                    | Kafka Topic |
                    +-------------+
                     /           \
                    v             v
        +-------------------+   +--------------------+
        | Alerting Consumer |   | Analytics Consumer |
        +-------------------+   +--------------------+
```


In [1]:
%pip install confluent-kafka psycopg2-binary

from confluent_kafka import Producer, Consumer
import psycopg2
import json
import time
from datetime import datetime
import uuid


Note: you may need to restart the kernel to use updated packages.


In [2]:
KAFKA_CONFIG = {
    "bootstrap.servers": "localhost:9092"
}

PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!"
}

print(f"Kafka bootstrap: {KAFKA_CONFIG['bootstrap.servers']}")


Kafka bootstrap: localhost:9092


## Topic Setup

A Kafka **topic** is the named stream where events are written. A topic is split into **partitions** so multiple producers and consumers can work in parallel. The **replication factor** controls how many copies of the data Kafka keeps across brokers for durability and availability. In this local lab, we use 3 partitions and replication factor 1 because there is only one broker.


In [3]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import KafkaException

admin = AdminClient(KAFKA_CONFIG)
topic_name = "citi.alerts"

existing_topics = admin.list_topics(timeout=10).topics
if topic_name in existing_topics:
    print("Topic citi.alerts already exists — OK")
else:
    topic = NewTopic(topic_name, num_partitions=3, replication_factor=1)
    futures = admin.create_topics([topic])
    try:
        futures[topic_name].result()
        print("Topic citi.alerts created (3 partitions, RF=1)")
    except Exception as exc:
        message = str(exc)
        if "TOPIC_ALREADY_EXISTS" in message or "already exists" in message.lower():
            print("Topic citi.alerts already exists — OK")
        else:
            raise


Topic citi.alerts already exists — OK


## Load Alerts from Postgres

We pull 100 HIGH/CRITICAL alerts from Postgres to produce into Kafka.


In [4]:
def load_alerts(limit=100):
    sql = '''
        SELECT
            alert_id,
            endpoint_id,
            severity,
            message,
            created_at
        FROM alerts
        WHERE severity IN ('HIGH', 'CRITICAL')
        ORDER BY created_at DESC
        LIMIT %s
    '''
    with psycopg2.connect(**PG_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute("SET search_path TO telemetry, public")
            cur.execute(sql, (limit,))
            rows = cur.fetchall()

    alerts = []
    for alert_id, endpoint_id, severity, message, created_at in rows:
        alerts.append(
            {
                "alert_id": alert_id,
                "endpoint_id": endpoint_id,
                "severity": severity,
                "message": message,
                "created_at": created_at.isoformat() if created_at is not None else None
            }
        )
    return alerts

alerts = load_alerts(limit=100)
print(f"Loaded {len(alerts)} alerts from Postgres")


Loaded 0 alerts from Postgres


## Producer

A Kafka **producer** publishes records into a topic. Each record can have a **key** and a **value**. Here, the key is the alert ID and the value is JSON. The delivery callback tells us whether Kafka acknowledged each message successfully.


In [5]:
def delivery_callback(err, msg):
    if err is not None:
        print(f"✗ delivery failed for key={msg.key().decode('utf-8') if msg.key() else None}: {err}")
    else:
        print(
            f"✓ delivered topic={msg.topic()} partition={msg.partition()} "
            f"offset={msg.offset()} key={msg.key().decode('utf-8') if msg.key() else None}"
        )

producer = Producer(KAFKA_CONFIG)

for alert in alerts:
    producer.produce(
        topic_name,
        key=str(alert["alert_id"]),
        value=json.dumps(alert),
        callback=delivery_callback
    )
    producer.poll(0)

producer.flush()
print(f"Produced {len(alerts)} alerts to {topic_name}")


Produced 0 alerts to citi.alerts


## Consumer

A Kafka **consumer** reads records from topics. Consumers join **consumer groups** so Kafka can distribute partitions across readers. An **offset** is the position of a message inside a partition. With `auto.offset.reset=earliest`, a brand-new consumer group starts from the beginning of the topic.


In [6]:
consumer = Consumer(
    {
        "bootstrap.servers": KAFKA_CONFIG["bootstrap.servers"],
        "group.id": f"notebook-group-1-{uuid.uuid4()}",
        "auto.offset.reset": "earliest"
    }
)

consumer.subscribe([topic_name])

consumed_messages = []
idle_start = time.time()

try:
    while True:
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            if time.time() - idle_start >= 3:
                break
            continue

        idle_start = time.time()

        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue

        payload = json.loads(msg.value().decode("utf-8"))
        consumed_messages.append(payload)

        if len(consumed_messages) >= 100:
            break
finally:
    consumer.close()

for i, payload in enumerate(consumed_messages[:5], start=1):
    print(f"Message {i}")
    print(json.dumps(payload, indent=2))
    print("-" * 60)

count = len(consumed_messages)
print(f"Consumed {count} messages from {topic_name}")


Consumed 0 messages from citi.alerts


## Offset Exploration

Offsets matter because they are Kafka's replay and recovery mechanism. A consumer does not just read "the topic" — it reads from a specific partition and offset. That offset acts like a bookmark. If a consumer crashes, Kafka can resume from the last committed offset instead of starting over.


In [7]:
from confluent_kafka import TopicPartition

admin = AdminClient(KAFKA_CONFIG)
metadata = admin.list_topics(topic=topic_name, timeout=10)
partitions = sorted(metadata.topics[topic_name].partitions.keys())

offset_consumer = Consumer(
    {
        "bootstrap.servers": KAFKA_CONFIG["bootstrap.servers"],
        "group.id": f"offset-inspector-{uuid.uuid4()}",
        "auto.offset.reset": "earliest"
    }
)

offset_lines = []
try:
    for partition_id in partitions:
        tp = TopicPartition(topic_name, partition_id)
        low, high = offset_consumer.get_watermark_offsets(tp, timeout=10)
        offset_lines.append(f"Partition {partition_id}: offset {high}")
finally:
    offset_consumer.close()

print(", ".join(offset_lines))


Partition 0: offset 0, Partition 1: offset 0, Partition 2: offset 0


## What Just Happened

- A Kafka topic acted as a **durable append-only log** for alert events.
- The producer wrote alert records once, and a consumer group read them independently without touching the producer.
- The **offset** served as a bookmark so consumers can replay or resume processing safely.
- The **replication factor** controls durability and availability; in this lab it is 1 because there is only one broker.
- Kafka beats a database queue here because it is designed for high-throughput fan-out, where many downstream consumers need the same stream.

This pattern scales to 6,000 endpoints emitting every 10 seconds = 600 events/sec. Kafka handles this trivially. A Postgres INSERT loop would not.


## Next Steps

- Run `kafka_concepts.md` to lock in the vocabulary
- T1-A2 is next — concept definitions before the deep dive in Round 2
